# ETL 4 mayoristas

## Librerias

In [1]:
import os
import argparse
import logging
import requests
import pickle
import gc
import unicodedata
import re
from datetime import datetime
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from psycopg2.extras import execute_batch
import time

from sqlalchemy.dialects.postgresql import insert

## Configuracion de Entorno

In [2]:
# Configuración inicial
load_dotenv()
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## Funcionalidades

### Normalizador de texto

In [5]:
def normalize_text(text_val: str) -> str:
    if not isinstance(text_val, str) or pd.isna(text_val):
        return ""
    text_val = text_val.lower()
    text_val = unicodedata.normalize('NFKD', text_val).encode('ASCII', 'ignore').decode('ASCII')
    text_val = re.sub(r'(\d+)\s*(ml|cm|mm|kg|g)', r'\1_\2', text_val)
    text_val = re.sub(r'[^a-z0-9\s\-_\/]', ' ', text_val)
    return text_val.strip()

### Conector Base de Datos

In [3]:
def get_db_engine():
    try:
        user = os.getenv("DB_USER")
        password = os.getenv("DB_PASS")
        host = os.getenv("DB_HOST")
        port = os.getenv("DB_PORT", "5432")
        database = os.getenv("DB_NAME")
        return create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}')
    except Exception as e:
        logger.error(f"Error en la conexión a la DB: {e}")
        return None

### Tipo de Cambio

In [6]:
def actualizar_tipo_cambio_usd(engine):
    logger.info("Consulta API Banxico: Actualizando tipo de cambio USD...")
    token = os.getenv("BANXICO_TOKEN")
    if not token:
        logger.error("Token de Banxico no encontrado.")
        return

    url = "https://www.banxico.org.mx/SieAPIRest/service/v1/series/SF43718/datos/oportuno"
    headers = {"Bmx-Token": token}

    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            data = response.json()
            dato = float(data['bmx']['series'][0]['datos'][0]['dato'])
            precio = int(dato) + (dato != int(dato)) 
            fecha_actualizacion = datetime.now()

            update_sql = "UPDATE tbl_cambio_divisas SET precio = :precio, fehca_actualizacion = :fecha_actualizacion WHERE divisa = :divisa;"
            insert_sql = "INSERT INTO tbl_cambio_divisas (divisa, precio, fehca_actualizacion) VALUES (:divisa, :precio, :fecha_actualizacion);"

            with engine.begin() as conn:
                result = conn.execute(text(update_sql), {"precio": precio, "fecha_actualizacion": fecha_actualizacion, "divisa": "USD"})
                if result.rowcount == 0:
                    conn.execute(text(insert_sql), {"divisa": "USD", "precio": precio, "fecha_actualizacion": fecha_actualizacion})
                    logger.info("✅ Registro de divisa insertado.")
                else:
                    logger.info("✅ Registro de divisa actualizado.")
        else:
            logger.error(f"Error Banxico: {response.status_code}")
    except Exception as e:
        logger.error(f"Error con Banxico: {e}")

### Algoritmo ponderacion

In [ ]:
def ponderacion_de_precio(engine):
    logger.info("Iniciando Ponderación de Precios Gaussiana...")
    try:
        df_div = pd.read_sql("SELECT divisa, precio FROM tbl_cambio_divisas", engine)
        tc = dict(zip(df_div['divisa'], df_div['precio']))

        query = """
            SELECT tdp.csku, tdp.cmoneda, tdp.nprecio, tdp.ndisponibilidad
            FROM tbl_detalle_producto tdp
            WHERE tdp.nprecio > 0 AND tdp.ndisponibilidad > 0
        """
        df = pd.read_sql(query, engine)
        
        if df.empty:
            logger.warning("No hay precios para ponderar.")
            return

        df['precio_mxn'] = df['cmoneda'].map(tc).fillna(1) * df['nprecio']
        resultados = []

        for sku, g in df.groupby('csku'):
            precios = g['precio_mxn'].values
            disponibilidad = g['ndisponibilidad'].values
            mu = precios.mean()
            sigma = precios.std()

            if sigma == 0: sigma = mu * 0.05

            peso_gauss = np.exp(-((precios - mu) ** 2) / (2 * sigma ** 2))
            peso_final = peso_gauss * disponibilidad
            sum_peso = np.sum(peso_final)
            
            costo = np.sum(precios * peso_final) / sum_peso if sum_peso > 0 else mu
            costo = float(round(costo * 1.05, 2))
            disponibilidad_total = int(disponibilidad.sum())
            resultados.append((costo, disponibilidad_total, sku))

        with engine.begin() as conn:
            raw_conn = conn.connection
            with raw_conn.cursor() as cursor:
                execute_batch(
                    cursor,
                    """
                    UPDATE tbl_producto 
                    SET nprecio_b2b = %s, ndisponibilidad_total = %s, tupdate_at = CURRENT_TIMESTAMP
                    WHERE csku = %s
                    """,
                    resultados, page_size=1000
                )
        logger.info(f"✅ Ponderación guardada para {len(resultados)} SKUs.")
    except Exception as e:
        logger.error(f"Error en ponderación: {e}")

### Estatus

In [7]:
def actualizar_estatus_productos(engine):
    logger.info("Actualizando Estatus (Activo/Inactivo)...")
    try:
        with engine.begin() as conn:
            res_desc = conn.execute(text("""
                UPDATE tbl_producto SET bestatus = false 
                WHERE ndisponibilidad_total = 0 OR csku NOT IN (SELECT DISTINCT csku FROM tbl_detalle_producto) OR ndisponibilidad_total IS NULL
            """))
            res_act = conn.execute(text("""
                UPDATE tbl_producto SET bestatus = true WHERE ndisponibilidad_total > 0
            """))
            logger.info(f"✅ Estatus actualizado. Desactivados: {res_desc.rowcount}, Activados: {res_act.rowcount}")
    except Exception as e:
        logger.error(f"Error estatus: {e}")

### Categorizador

In [8]:
def categorizador_deep_learning(engine):
    logger.info("🧠 Iniciando Categorizador Deep Learning...")
    try:
        import tensorflow as tf
        from tensorflow.keras.utils import pad_sequences
        
        query = "SELECT csku, cnombre, cdescripcion, cmarca FROM tbl_producto WHERE nid_subcategoria IS NULL;"
        df_productos = pd.read_sql(query, engine)
        
        if df_productos.empty: return

        BASE_DIR = os.path.abspath(os.getcwd())
        path_modelo = os.path.join(BASE_DIR, "Red_neuronal", "modelo_categorias.keras")
        path_tokenizer = os.path.join(BASE_DIR, "Red_neuronal", "tokenizer.pkl")
        path_labelencoder = os.path.join(BASE_DIR, "Red_neuronal", "labelencoder.pkl")
        
        model = tf.keras.models.load_model(path_modelo)
        with open(path_tokenizer, "rb") as f: tokenizer = pickle.load(f)
        with open(path_labelencoder, "rb") as f: le = pickle.load(f)
            
        df_productos["texto"] = (df_productos["cnombre"].fillna("") + " " + df_productos["cdescripcion"].fillna("") + " " + df_productos["cmarca"].fillna(""))
        df_productos["texto_norm"] = df_productos["texto"].apply(normalize_text)
        
        secuencias = tokenizer.texts_to_sequences(df_productos["texto_norm"])
        X = pad_sequences(secuencias, maxlen=300)
        
        y_pred = model.predict(X, batch_size=32)
        y_classes = np.argmax(y_pred, axis=1)
        df_productos["categoria_predicha"] = le.inverse_transform(y_classes)
        
        df_subcategorias = pd.read_sql("SELECT nid as id_subcategoria_nueva, cnombre_subcategoria FROM tbl_subcategoria", engine)
        df_productos = df_productos.merge(df_subcategorias, left_on="categoria_predicha", right_on="cnombre_subcategoria", how="left")
        
        updates = [u for u in zip(df_productos["id_subcategoria_nueva"], df_productos["csku"]) if pd.notna(u[0])]
        
        if updates:
            with engine.begin() as conn:
                raw_conn = conn.connection
                with raw_conn.cursor() as cursor:
                    execute_batch(cursor, "UPDATE tbl_producto SET nid_subcategoria = %s WHERE csku = %s;", updates, page_size=1000)
            logger.info(f"✅ {len(updates)} productos categorizados con Deep Learning.")
            
    except Exception as e:
        logger.error(f"Error NLP: {e}")
    finally:
        if 'model' in locals(): del model
        if 'tokenizer' in locals(): del tokenizer
        if 'le' in locals(): del le
        if 'df_productos' in locals(): del df_productos
        if 'tf' in locals(): tf.keras.backend.clear_session()
        gc.collect()

## Main

In [10]:
engine = get_db_engine()
# Consulta y creación del DataFrame de CT
try:
    query = "SELECT * FROM temp_tbl_ct;"
    df_ct = pd.read_sql(query, engine)
    print('Catalogo de CT Obtenido')
except Exception as e:
    print("Error al ejecutar la consulta:", e)
        
# Consulta y creación del DataFrame de Exel
try:
    query = "SELECT * FROM temp_tbl_exel;"
    df_exel = pd.read_sql(query, engine)
    print('Catalogo de exel Obtenido')
except Exception as e:
    print("Error al ejecutar la consulta:", e)    
        
# Consulta y creación del DataFrame de CVA
try:
    query = "SELECT * FROM temp_tbl_cva;"
    df_cva = pd.read_sql(query, engine)
    print('Catalogo de cva Obtenido')
except Exception as e:
    print("Error al ejecutar la consulta:", e) 
        
# Consulta y creación del DataFrame de Syscom
try:
    query = "SELECT * FROM temp_tbl_syscom;"
    df_syscom = pd.read_sql(query, engine)
    print('Catalogo de syscom Obtenido')
except Exception as e:
    print("Error al ejecutar la consulta:", e)
        
engine.dispose()

Catalogo de CT Obtenido
Catalogo de exel Obtenido
Catalogo de cva Obtenido
Catalogo de syscom Obtenido


In [ ]:
# 1. Configuración de Prioridades y Nulos
PRIORIDAD = ['exel', 'syscom', 'cva', 'ct']
CAMPOS_A_CONSOLIDAR = ["nombre", "marca", "categoria", "descripcion", "especificaciones", "imagen"]
VALORES_NULOS = ["NULL", "null", "None", "", "ND"]

In [12]:
df_dict = {
    'ct': df_ct,
    'exel': df_exel,
    'cva': df_cva,
    'syscom': df_syscom,
}

In [13]:
def preparar_y_limpiar_df(df, prov):
    """Limpia nulos, elimina ID_PROVEEDOR para evitar colisiones y asienta el SKU como índice."""
    df = df.copy()
    
    # Asegurar que el SKU sea texto limpio y no tenga duplicados
    if 'SKU' in df.columns:
        df['SKU'] = df['SKU'].astype(str).str.strip()
        df = df.drop_duplicates(subset=['SKU'])
    else:
        return pd.DataFrame() # Si no hay SKU, descartamos
        
    # Eliminar ID_PROVEEDOR ya que causará colisión en el concat y no tiene sufijo
    if 'ID_PROVEEDOR' in df.columns:
        df = df.drop(columns=['ID_PROVEEDOR'])

    # Limpiar nulos solo en columnas de texto
    cols_texto = df.select_dtypes(include=['object', 'string']).columns
    df[cols_texto] = df[cols_texto].replace(VALORES_NULOS, np.nan)
    
    # Devolver DataFrame con el SKU como índice para una unión ultra rápida
    return df.set_index('SKU')

In [14]:
# 2. Preparar todos los DataFrames
dfs_preparados = [preparar_y_limpiar_df(df_dict[prov], prov) 
                  for prov in PRIORIDAD if prov in df_dict]

In [15]:
# 3. Unión Simultánea (Outer Join optimizado)
# Une todo rápidamente usando el SKU como índice
df_unido = pd.concat(dfs_preparados, axis=1)

In [16]:
# 4. Consolidación y Auditoría
for campo in CAMPOS_A_CONSOLIDAR:
    # Buscar las columnas exactas que ya traen el sufijo en tu DataFrame
    cols_campo = [f"{campo}_{prov}" for prov in PRIORIDAD if f"{campo}_{prov}" in df_unido.columns]
    
    if not cols_campo:
        continue
        
    df_temp = df_unido[cols_campo]
    
    # 1. Consolidar el valor tomando el de mayor prioridad
    df_unido[campo] = df_temp.bfill(axis=1).iloc[:, 0]
    
    # 2. Saber si al menos un proveedor tiene información en esta fila (retorna True o False)
    tiene_dato = df_temp.notna().any(axis=1)
    
    # 3. Extraer el nombre del proveedor (idxmax falla si todos son nulos, pero lo corregimos en el paso 4)
    fuente_cruda = df_temp.notna().idxmax(axis=1).str.replace(f"{campo}_", "", regex=False)
    
    # 4. CORRECCIÓN: Asignar la fuente SOLO si 'tiene_dato' es True, de lo contrario asignar np.nan
    df_unido[f"{campo}_fuente"] = np.where(tiene_dato, fuente_cruda, np.nan)
    
    # Liberar memoria eliminando las columnas originales del proveedor
    df_unido.drop(columns=cols_campo, inplace=True)

In [17]:
# 5. Limpieza Final y Formateo
df_final = df_unido.reset_index() # Recuperamos el SKU a una columna normal
df_final = df_final.dropna(subset=['SKU', 'nombre']) # Borramos si no hay SKU o si la consolidación de nombre dio nulo

In [18]:
# Mapeo de nombres finales
mapa_columnas = {
    'SKU': 'csku',
    'nombre': 'cnombre',
    'categoria': 'ccategoria',
    'marca': 'cmarca',
    'descripcion': 'cdescripcion',
    'especificaciones': 'cespecificaciones',
    'imagen': 'cimagen'
}

In [19]:
df_final = df_final.rename(columns=mapa_columnas)
df_final['bestatus'] = True

In [20]:
df_final.info()

<class 'pandas.DataFrame'>
Index: 60386 entries, 0 to 60386
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   csku                     60386 non-null  str    
 1   disponibilidad_exel      14843 non-null  float64
 2   precio_exel              14843 non-null  float64
 3   moneda_exel              14843 non-null  str    
 4   clave_producto_exel      14843 non-null  str    
 5   disponibilidad_syscom    36670 non-null  float64
 6   precio_syscom            36670 non-null  float64
 7   moneda_syscom            36670 non-null  str    
 8   clave_producto_syscom    36670 non-null  str    
 9   disponibilidad_cva       9908 non-null   float64
 10  precio_cva               9908 non-null   float64
 11  moneda_cva               9908 non-null   str    
 12  clave_producto_cva       9908 non-null   str    
 13  disponibilidad_ct        5924 non-null   float64
 14  precio_ct                5924 non-null

In [21]:
cols_finales = [
    'csku', 'cnombre', 'cmarca', 
     'cdescripcion',
    'cespecificaciones',  'cimagen',  'bestatus'
]

In [22]:
# Protegemos contra posibles columnas faltantes
cols_existentes = [c for c in cols_finales if c in df_final.columns]
df_tbl_productos = df_final[cols_existentes].copy()

In [24]:
df_tbl_productos['tcreate_at'] = datetime.now()
df_tbl_productos['tupdate_at'] = datetime.now()
df_tbl_productos['bestatus'] = True

In [25]:
cols_finale_det = [ 'csku','disponibilidad_exel','precio_exel','moneda_exel','clave_producto_exel',
                   'disponibilidad_syscom','precio_syscom','moneda_syscom','clave_producto_syscom',
                   'disponibilidad_cva','precio_cva','moneda_cva','clave_producto_cva',
                   'disponibilidad_ct','precio_ct','moneda_ct','clave_producto_ct',
                   'disponibilidad_dcm','precio_dcm','moneda_dcm','clave_producto_dcm'
]

In [26]:
# Protegemos contra posibles columnas faltantes
cols_existentes = [c for c in cols_finale_det if c in df_final.columns]
df_tbl_det = df_final[cols_existentes].copy()

In [27]:
# Lista para almacenar los bloques de datos limpios
dfs_detalles = []

# Iteramos directamente sobre tu diccionario original de DataFrames (df_dict)
for prov, df_origen in df_dict.items():
    
    # Armamos la lista de las columnas exactas que necesitamos de este proveedor
    cols_necesarias = [
        'SKU', 
        'ID_PROVEEDOR', 
        f'disponibilidad_{prov}', 
        f'moneda_{prov}', 
        f'precio_{prov}', 
        f'clave_producto_{prov}'
    ]
    
    # Validamos que el DataFrame original contenga todas estas columnas
    if all(col in df_origen.columns for col in cols_necesarias):
        
        # Extraemos solo esa "rebanada" de columnas
        df_temp = df_origen[cols_necesarias].copy()
        
        # Renombramos directamente a tu estructura final
        # Como las extrajimos en orden, podemos renombrarlas directamente por posición
        df_temp.columns = [
            'csku', 
            'nid_proveedor', 
            'ndisponibilidad', 
            'cmoneda', 
            'nprecio', 
            'cclave_producto'
        ]
        
        # Limpieza de SKU
        df_temp['csku'] = df_temp['csku'].astype(str).str.strip()
        
        
        # Lo guardamos en la lista
        dfs_detalles.append(df_temp)

# Unimos todos los proveedores verticalmente en un solo paso súper rápido
df_final_det = pd.concat(dfs_detalles, ignore_index=True)

# Aseguramos los tipos de datos correctos
df_final_det['ndisponibilidad'] = df_final_det['ndisponibilidad'].astype(int)
df_final_det['nprecio'] = df_final_det['nprecio'].astype(float)
df_final_det['nid_proveedor'] = df_final_det['nid_proveedor'].astype(int)

In [28]:
start_time = time.time()

In [ ]:
try:
    # La clave aquí es method='multi' junto con un chunksize mayor
    df_tbl_productos.to_sql(
        name='tbl_producto', 
        con=engine, 
        if_exists='append', 
        index=False, 
        chunksize=5000, 
        method='multi'
    )
    
    tiempo_total = round(time.time() - start_time, 2)
    logger.info(f"Insertados {len(df_tbl_productos)} productos masivamente en {tiempo_total} segundos.")

except Exception as e:
    logger.error(f"Error en la subida masiva de productos: {e}")

In [ ]:
# 1. ALEATORIEDAD: Mezclamos el DataFrame. 
# Si hay varios registros con la misma disponibilidad máxima, al estar mezclados, 
# el que quede arriba después del sorteo será "al azar".
df_final_det = df_final_det.sample(frac=1).reset_index(drop=True)

# 2. ORDENAMIENTO ESTRATÉGICO:
# Ordenamos por SKU y Proveedor (ascendente) y por disponibilidad (descendente).
# Esto pone el registro con mayor 'ndisponibilidad' al principio de cada grupo.
df_final_det = df_final_det.sort_values(
    by=['csku', 'nid_proveedor', 'ndisponibilidad'], 
    ascending=[True, True, False]
)

# 3. ELIMINACIÓN DE DUPLICADOS:
# 'keep=first' mantendrá solo la primera fila de cada combinación SKU+Proveedor.
# Como ordenamos por disponibilidad descendente, esa primera fila es la que tiene más stock.
df_final_det = df_final_det.drop_duplicates(
    subset=['csku', 'nid_proveedor'], 
    keep='first'
)
# Sobreescribimos el DataFrame conservando solo los SKUs que NO son 'NULL'
df_final_det = df_final_det[df_final_det['csku'] != 'NULL']

In [ ]:
# 1. Definimos la función de Upsert adaptada a tus llaves
def upsert_detalles(table, conn, keys, data_iter):
    """Inserta nuevos precios o actualiza los existentes si la llave compuesta ya existe."""
    data = [dict(zip(keys, row)) for row in data_iter]
    stmt = insert(table.table).values(data)
    
    # Aquí le decimos a Python cuáles son las dos llaves amarillas de tu imagen
    llaves_unicas = ['csku', 'nid_proveedor']
    
    # Si la combinación SKU+Proveedor ya existe, le decimos que actualice el resto de columnas
    update_dict = {c.name: c for c in stmt.excluded if c.name not in llaves_unicas}
    
    on_conflict_stmt = stmt.on_conflict_do_update(
        index_elements=llaves_unicas, 
        set_=update_dict
    )
    
    conn.execute(on_conflict_stmt)

# 2. Ejecutamos la subida masiva
start_time = time.time()

try:
    df_final_det.to_sql(
        name='tbl_detalle_producto',  # Asegúrate de que el nombre de la tabla sea correcto
        con=engine, 
        if_exists='append', 
        index=False, 
        chunksize=10000, 
        method=upsert_detalles # Inyectamos la magia aquí
    )
    
    tiempo_total = round(time.time() - start_time, 2)
    logger.info(f"Upsert exitoso: Se procesaron {len(df_final_det)} detalles en {tiempo_total} segundos.")

except Exception as e:
    logger.error(f"Error en el Upsert de detalles: {e}")

In [ ]:
# 1. Encontrar todos los registros con la combinación csku + nid_proveedor repetida
df_duplicados = df_final_det[df_final_det.duplicated(subset=['csku', 'nid_proveedor'], keep=False)]

# 2. Ordenar para que los duplicados aparezcan juntos
df_duplicados = df_duplicados.sort_values(by=['nid_proveedor', 'csku'])

# 3. Tu mapeo exacto de IDs a Nombres
mapa_nombres = {
    4: 'EXEL',
    2: 'SYSCOM',
    1: 'CVA',
    3: 'CT',
    5: 'DCM'
}

# Crear una columna temporal solo para ver el nombre en la auditoría
df_duplicados['nombre_proveedor'] = df_duplicados['nid_proveedor'].map(mapa_nombres)

# 4. Imprimir el resumen
print("--- RESUMEN DE DUPLICADOS POR PROVEEDOR ---")
resumen = df_duplicados['nombre_proveedor'].value_counts()
print(resumen)
print(f"\nTotal de filas conflictivas: {len(df_duplicados)}")

# 5. Ver el detalle visual
print("\n--- DETALLE DE LOS DUPLICADOS ---")
columnas_ver = ['nombre_proveedor', 'nid_proveedor', 'csku', 'ndisponibilidad', 'nprecio', 'cmoneda']
print(df_duplicados[columnas_ver].head(30).to_string(index=False))